# Controle Romance: Espanhol

Processa 1M tokens de espanhol (FineWeb-2 + Wikipedia) na layer 13 e compara
com os stats PT/EN existentes para distinguir "PT-specific" de "Romance-specific."

Para cada feature PT-specific, calcula:
- LSI_es = (freq_PT - freq_ES) / (freq_PT + freq_ES)  → positivo = PT > ES
- Se feature ativa igualmente em ES → Romance-specific
- Se feature ativa muito mais em PT → genuinamente PT-specific

**Tempo:** ~15 min no T4 | **Output:** `exp_controle_espanhol_results.json`

In [ ]:
!pip install transformer_lens sae_lens datasets -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "./data/checkpoints/"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

## Setup: Modelo + SAE (Layer 13)

In [ ]:
import torch
import numpy as np
import json
import time
from tqdm import tqdm
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LAYER = 13
N_TOKENS = 1_000_000
SEQ_LEN = 128
BATCH_SIZE = 4
MIN_ACTS = 100
LSI_THRESHOLD = 0.3
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

from sae_lens import SAE, HookedSAETransformer
from datasets import load_dataset

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b", device=device, dtype=torch.float16,
)
tokenizer = model.tokenizer
print(f"Modelo: {model.cfg.model_name}")

print(f"Carregando SAE: {SAE_ID}...")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
print(f"SAE: {sae.cfg.d_sae} features")

## Coleta de tokens espanhóis (Wikipedia ES + FineWeb-2 ES)

In [ ]:
def collect_tokens(dataset, n_tokens, text_field="text", desc="Tokenizando"):
    all_ids = []
    n_articles = 0
    for article in tqdm(dataset, desc=desc):
        text = article[text_field]
        if not text or len(text) < 50:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_ids.extend(ids)
        n_articles += 1
        if len(all_ids) >= n_tokens:
            break
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_articles} textos -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs)")
    return tokens

print("Wikipedia ES...")
wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)
tokens_wiki_es = collect_tokens(wiki_es, N_TOKENS, desc="Wiki ES")

print("FineWeb-2 ES...")
web_es = load_dataset("HuggingFaceFW/fineweb-2", "spa_Latn", split="train", streaming=True)
tokens_web_es = collect_tokens(web_es, N_TOKENS, desc="FineWeb ES")

print(f"\nTotal ES: {tokens_wiki_es.numel() + tokens_web_es.numel():,} tokens")

## Compute feature stats para espanhol

In [ ]:
def compute_feature_stats(model, sae, tokens, hook_name, batch_size=BATCH_SIZE, desc=""):
    n_features = sae.cfg.d_sae
    counts = torch.zeros(n_features, device=device)
    sums = torch.zeros(n_features, device=device)
    maxvals = torch.zeros(n_features, device=device)
    total = 0

    n_batches = (len(tokens) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(tokens), batch_size), desc=desc, total=n_batches):
        batch = tokens[i:i+batch_size].to(device)
        with torch.no_grad():
            _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook_name)
            acts = cache[hook_name]
            feat_acts = sae.encode(acts)

            active = feat_acts > 0
            counts += active.float().sum(dim=(0, 1))
            sums += feat_acts.sum(dim=(0, 1))
            maxvals = torch.max(maxvals, feat_acts.amax(dim=(0, 1)))
            total += batch.numel()

        del cache, acts, feat_acts, active, batch
        torch.cuda.empty_cache()

    print(f"  {total:,} tokens | {(counts > 0).sum().item():,} features ativas")
    return {"counts": counts.cpu(), "sums": sums.cpu(), "max": maxvals.cpu(), "total_tokens": total}

t0 = time.time()

print("Processando Wikipedia ES...")
stats_wiki_es = compute_feature_stats(model, sae, tokens_wiki_es, HOOK_NAME, desc="Wiki ES")

print("Processando FineWeb ES...")
stats_web_es = compute_feature_stats(model, sae, tokens_web_es, HOOK_NAME, desc="FineWeb ES")

elapsed = time.time() - t0
print(f"\nTempo total: {elapsed:.0f}s ({elapsed/60:.1f} min)")

torch.save({
    "stats_wiki_es": stats_wiki_es,
    "stats_web_es": stats_web_es,
}, os.path.join(SAVE_DIR, "stats_espanhol.pt"))
print("Checkpoint salvo: stats_espanhol.pt")

## Comparação PT vs ES vs EN

Carrega stats PT/EN dos checkpoints existentes e compara com espanhol.

In [ ]:
print("Carregando stats PT/EN dos checkpoints existentes...")
stats_wiki_pt = torch.load(os.path.join(SAVE_DIR, "stats_wiki_pt.pt"), weights_only=False)
stats_wiki_en = torch.load(os.path.join(SAVE_DIR, "stats_wiki_en.pt"), weights_only=False)
stats_mc4_pt  = torch.load(os.path.join(SAVE_DIR, "stats_mc4_pt.pt"),  weights_only=False)
stats_c4_en   = torch.load(os.path.join(SAVE_DIR, "stats_c4_en.pt"),   weights_only=False)
print("OK")

# Frequências PT (combinada wiki + web)
freq_wiki_pt = stats_wiki_pt["counts"] / stats_wiki_pt["total_tokens"]
freq_wiki_en = stats_wiki_en["counts"] / stats_wiki_en["total_tokens"]
freq_mc4_pt  = stats_mc4_pt["counts"]  / stats_mc4_pt["total_tokens"]
freq_c4_en   = stats_c4_en["counts"]   / stats_c4_en["total_tokens"]

freq_pt = (freq_wiki_pt + freq_mc4_pt) / 2
freq_en = (freq_wiki_en + freq_c4_en) / 2

# Frequências ES (combinada wiki + web)
freq_wiki_es = stats_wiki_es["counts"] / stats_wiki_es["total_tokens"]
freq_web_es  = stats_web_es["counts"]  / stats_web_es["total_tokens"]
freq_es = (freq_wiki_es + freq_web_es) / 2

# LSI original (PT vs EN)
lsi_pt_en = (freq_pt - freq_en) / (freq_pt + freq_en + 1e-10)

# LSI PT vs ES
lsi_pt_es = (freq_pt - freq_es) / (freq_pt + freq_es + 1e-10)

# LSI ES vs EN
lsi_es_en = (freq_es - freq_en) / (freq_es + freq_en + 1e-10)

# Features ativas
total_counts = (stats_wiki_pt["counts"] + stats_wiki_en["counts"]
                + stats_mc4_pt["counts"] + stats_c4_en["counts"]
                + stats_wiki_es["counts"] + stats_web_es["counts"])
active = total_counts >= MIN_ACTS
n_active = active.sum().item()
print(f"\nFeatures ativas (≥{MIN_ACTS}): {n_active:,}")

# Classificação
pt_specific = (lsi_pt_en > LSI_THRESHOLD) & active
es_specific = (lsi_es_en > LSI_THRESHOLD) & active
en_specific = (lsi_pt_en < -LSI_THRESHOLD) & active

pt_not_es = pt_specific & (lsi_pt_es > LSI_THRESHOLD)   # PT >> ES (genuinamente PT)
pt_and_es = pt_specific & (lsi_pt_es.abs() <= LSI_THRESHOLD)  # PT ≈ ES (Romance)
pt_less_es = pt_specific & (lsi_pt_es < -LSI_THRESHOLD)  # ES >> PT

print(f"\n{'='*60}")
print("CLASSIFICAÇÃO DAS FEATURES PT-SPECIFIC (LSI_pt_en > 0.3)")
print(f"{'='*60}")
print(f"  Total PT-specific (vs EN):        {pt_specific.sum().item():,}")
print(f"  → Genuinamente PT (PT >> ES):     {pt_not_es.sum().item():,}")
print(f"  → Romance-specific (PT ≈ ES):     {pt_and_es.sum().item():,}")
print(f"  → Mais ES que PT (ES >> PT):      {pt_less_es.sum().item():,}")
print(f"\n  ES-specific (vs EN):              {es_specific.sum().item():,}")

# Proporções
n_pt = pt_specific.sum().item()
if n_pt > 0:
    print(f"\n  % genuinamente PT:  {pt_not_es.sum().item()/n_pt*100:.1f}%")
    print(f"  % Romance-specific: {pt_and_es.sum().item()/n_pt*100:.1f}%")
    print(f"  % mais ES que PT:   {pt_less_es.sum().item()/n_pt*100:.1f}%")

## Features validadas por probes: PT-specific ou Romance-specific?

In [ ]:
PROBE_FEATURES = {
    "Gender (1215)":          1215,
    "Crase (4584)":           4584,
    "Enclisis (2817)":        2817,
    "Proclisis (6215)":       6215,
    "Contraction do (2294)":  2294,
    "Clitic lhe (15135)":    15135,
    "Preposition por (10478)":10478,
    "Personal inf (10349)":  10349,
}

TOP_FEATURES = {
    "PT detector (10496)":    10496,
    "Narrative PT (1906)":     1906,
    "Encyclopedic PT (8718)":  8718,
    "Articles PT (16057)":    16057,
    "Metadata PT (12649)":    12649,
    "Connectors PT (16194)":  16194,
    "Colloquial (1082)":       1082,
    "Scientific (5880)":       5880,
}

print(f"{'='*80}")
print("FEATURES VALIDADAS POR PROBES")
print(f"{'='*80}")
print(f"  {'Feature':<28} {'freq_PT':>8} {'freq_ES':>8} {'freq_EN':>8} {'LSI_pt_en':>10} {'LSI_pt_es':>10} {'Classe':>16}")
print(f"  {'-'*94}")

probe_results = {}
for name, fid in PROBE_FEATURES.items():
    fp = freq_pt[fid].item()
    fe = freq_es[fid].item()
    fn = freq_en[fid].item()
    lpe = lsi_pt_en[fid].item()
    lps = lsi_pt_es[fid].item()

    if lps > LSI_THRESHOLD:
        classe = "PT-specific"
    elif lps > -LSI_THRESHOLD:
        classe = "Romance-shared"
    else:
        classe = "ES-dominant"

    probe_results[name] = {
        "feature_id": fid,
        "freq_pt": round(fp, 6),
        "freq_es": round(fe, 6),
        "freq_en": round(fn, 6),
        "lsi_pt_en": round(lpe, 4),
        "lsi_pt_es": round(lps, 4),
        "classification": classe,
    }
    print(f"  {name:<28} {fp:>8.5f} {fe:>8.5f} {fn:>8.5f} {lpe:>+10.4f} {lps:>+10.4f} {classe:>16}")

print(f"\n{'='*80}")
print("TOP FEATURES PT-SPECIFIC")
print(f"{'='*80}")
print(f"  {'Feature':<28} {'freq_PT':>8} {'freq_ES':>8} {'freq_EN':>8} {'LSI_pt_en':>10} {'LSI_pt_es':>10} {'Classe':>16}")
print(f"  {'-'*94}")

top_results = {}
for name, fid in TOP_FEATURES.items():
    fp = freq_pt[fid].item()
    fe = freq_es[fid].item()
    fn = freq_en[fid].item()
    lpe = lsi_pt_en[fid].item()
    lps = lsi_pt_es[fid].item()

    if lps > LSI_THRESHOLD:
        classe = "PT-specific"
    elif lps > -LSI_THRESHOLD:
        classe = "Romance-shared"
    else:
        classe = "ES-dominant"

    top_results[name] = {
        "feature_id": fid,
        "freq_pt": round(fp, 6),
        "freq_es": round(fe, 6),
        "freq_en": round(fn, 6),
        "lsi_pt_en": round(lpe, 4),
        "lsi_pt_es": round(lps, 4),
        "classification": classe,
    }
    print(f"  {name:<28} {fp:>8.5f} {fe:>8.5f} {fn:>8.5f} {lpe:>+10.4f} {lps:>+10.4f} {classe:>16}")

# Resumo
n_pt_only = sum(1 for v in probe_results.values() if v["classification"] == "PT-specific")
n_romance = sum(1 for v in probe_results.values() if v["classification"] == "Romance-shared")
n_es_dom = sum(1 for v in probe_results.values() if v["classification"] == "ES-dominant")
print(f"\n  Probe features: {n_pt_only} PT-specific, {n_romance} Romance-shared, {n_es_dom} ES-dominant")

## Correlação geral e salvar resultados

In [ ]:
mask = active.numpy()

# Correlações
rho_pt_es, p_pt_es = spearmanr(lsi_pt_en.numpy()[mask], lsi_es_en.numpy()[mask])
rho_freq, p_freq = spearmanr(freq_pt.numpy()[mask], freq_es.numpy()[mask])

print(f"{'='*60}")
print("CORRELAÇÕES GLOBAIS")
print(f"{'='*60}")
print(f"  Spearman LSI_pt_en vs LSI_es_en: ρ = {rho_pt_es:.4f} (p = {p_pt_es:.2e})")
print(f"  Spearman freq_PT vs freq_ES:     ρ = {rho_freq:.4f} (p = {p_freq:.2e})")

# Distribuição de LSI_pt_es para features PT-specific
pt_mask = pt_specific.numpy()
lsi_pt_es_vals = lsi_pt_es.numpy()[pt_mask]
print(f"\n{'='*60}")
print(f"DISTRIBUIÇÃO LSI_pt_es PARA {pt_mask.sum()} FEATURES PT-SPECIFIC")
print(f"{'='*60}")
print(f"  Mean:   {lsi_pt_es_vals.mean():.4f}")
print(f"  Median: {np.median(lsi_pt_es_vals):.4f}")
print(f"  Std:    {lsi_pt_es_vals.std():.4f}")
print(f"  Min:    {lsi_pt_es_vals.min():.4f}")
print(f"  Max:    {lsi_pt_es_vals.max():.4f}")

# Histograma textual
bins = [-1.0, -0.5, -0.3, 0.0, 0.3, 0.5, 1.0]
hist, _ = np.histogram(lsi_pt_es_vals, bins=bins)
print(f"\n  Distribuição LSI_pt_es:")
for i in range(len(bins)-1):
    bar = "█" * (hist[i] // 5)
    print(f"    [{bins[i]:+.1f}, {bins[i+1]:+.1f}): {hist[i]:>4} {bar}")

# Salvar
ALL_RESULTS = {
    "experiment": "Controle Romance — Espanhol",
    "model": "gemma-2-2b",
    "layer": LAYER,
    "n_tokens_es_per_corpus": N_TOKENS,
    "summary": {
        "n_active": n_active,
        "pt_specific_vs_en": int(pt_specific.sum().item()),
        "genuinely_pt": int(pt_not_es.sum().item()),
        "romance_shared": int(pt_and_es.sum().item()),
        "es_dominant": int(pt_less_es.sum().item()),
        "es_specific_vs_en": int(es_specific.sum().item()),
        "pct_genuinely_pt": round(pt_not_es.sum().item()/max(pt_specific.sum().item(),1)*100, 1),
        "pct_romance_shared": round(pt_and_es.sum().item()/max(pt_specific.sum().item(),1)*100, 1),
    },
    "correlations": {
        "lsi_pt_en_vs_lsi_es_en": {"rho": round(rho_pt_es, 4), "pval": float(p_pt_es)},
        "freq_pt_vs_freq_es": {"rho": round(rho_freq, 4), "pval": float(p_freq)},
    },
    "lsi_pt_es_distribution": {
        "mean": round(float(lsi_pt_es_vals.mean()), 4),
        "median": round(float(np.median(lsi_pt_es_vals)), 4),
        "std": round(float(lsi_pt_es_vals.std()), 4),
    },
    "probe_features": probe_results,
    "top_features": top_results,
}

output_path = os.path.join(SAVE_DIR, "exp_controle_espanhol_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ALL_RESULTS, f, ensure_ascii=False, indent=2)

print(f"\nResultados salvos em: {output_path}")
print("Pronto! Baixe o exp_controle_espanhol_results.json do Drive.")